# Proyecto inter

geometry.py:Define las entidades Point y Square requeridas para la representación espacial
y cálculo de contención/intersección en 2D.

In [10]:
"""Módulo de primitivas geométricas para la arquitectura del Quadtree.

Define las entidades Point y Square requeridas para la representación espacial
y cálculo de contención/intersección en 2D.
"""

from __future__ import annotations

from typing import Tuple


class Point:
    """Representa un punto en el espacio euclidiano bidimensional."""

    def __init__(self, x: float, y: float) -> None:
        """Inicializa un punto con coordenadas (x, y).

        Args:
            x: Coordenada en el eje horizontal.
            y: Coordenada en el eje vertical.
        """
        self.x = float(x)
        self.y = float(y)

    def distance_to(self, other: Point) -> float:
        """Calcula la distancia euclidiana hacia otro punto."""
        return ((self.x - other.x) ** 2 + (self.y - other.y) ** 2) ** 0.5

    def __repr__(self) -> str:
        return f"Point(x={self.x:.4f}, y={self.y:.4f})"


class Square:
    """Representa una región espacial cuadrada delimitada por sus coordenadas.

    Sigue la terminología y formalización de la presentación: esquinas,
    aristas, centro y longitud de lado.
    """

    def __init__(self, x_mid: float, y_mid: float, side_length: float) -> None:
        """Inicializa el cuadrado a partir de su centro y longitud de lado.

        Args:
            x_mid: Coordenada X del punto central (Xmid).
            y_mid: Coordenada Y del punto central (Ymid).
            side_length: Longitud total del lado (side).
        """
        self.x_mid = float(x_mid)
        self.y_mid = float(y_mid)
        self.side_length = float(side_length)
        self.half_side = self.side_length / 2.0

        # Límites cartesianos mínimos y máximos
        self.x_min = self.x_mid - self.half_side
        self.x_max = self.x_mid + self.half_side
        self.y_min = self.y_mid - self.half_side
        self.y_max = self.y_mid + self.half_side

    @property
    def corners(self) -> dict[str, Tuple[float, float]]:
        """Devuelve las coordenadas de las cuatro esquinas (corners)."""
        return {
            "SW": (self.x_min, self.y_min),
            "SE": (self.x_max, self.y_min),
            "NW": (self.x_min, self.y_max),
            "NE": (self.x_max, self.y_max),
        }

    @property
    def edges(self) -> dict[str, Tuple[Tuple[float, float], Tuple[float, float]]]:
        """Devuelve los segmentos correspondientes a las cuatro aristas (edges)."""
        c = self.corners
        return {
            "south": (c["SW"], c["SE"]),
            "north": (c["NW"], c["NE"]),
            "west": (c["SW"], c["NW"]),
            "east": (c["SE"], c["NE"]),
        }

    def contains(self, point: Point) -> bool:
        """Verifica si un punto se encuentra dentro de los límites del cuadrado.

        Se utiliza el criterio semicerrado en bordes internos para evitar duplicidad
        en particiones contiguas: [x_min, x_max) y [y_min, y_max). En el borde
        externo máximo del dominio completo se admite inclusión.
        """
        return (self.x_min <= point.x <= self.x_max) and (
            self.y_min <= point.y <= self.y_max
        )

    def intersects(self, other: Square) -> bool:
        """Verifica si este cuadrado interseca con otra región cuadrada."""
        return not (
            other.x_min > self.x_max
            or other.x_max < self.x_min
            or other.y_min > self.y_max
            or other.y_max < self.y_min
        )

    def __repr__(self) -> str:
        return (
            f"Square(center=({self.x_mid:.2f}, {self.y_mid:.2f}), "
            f"side={self.side_length:.2f})"
        )

quadtree.py:Aplica el paradigma Divide y Vencerás mediante subdivisión recursiva de cuadrantes
y acota la profundidad máxima basada en la distancia mínima entre puntos.

In [ ]:
"""Implementación de la estructura de datos Quadtree orientada a partición 2D.
"""

from __future__ import annotations

import math
from typing import List, Optional
from geometry import Point, Square


class QuadNode:
    """Nodo del Quadtree con los atributos base de la arquitectura técnica."""

    def __init__(self, boundary: Square, capacity: int, depth: int = 0) -> None:
        """Inicializa un nodo del Quadtree.

        Args:
            boundary: Región espacial (Square) que ocupa el nodo.
            capacity: Límite máximo de elementos antes de dividir el nodo.
            depth: Profundidad actual del nodo en el árbol (raíz = 0).
        """
        self.boundary: Square = boundary
        self.capacity: int = int(capacity)
        self.depth: int = int(depth)

        # Lista de datos/puntos almacenados directamente en este nodo
        self.points: List[Point] = []

        # Estado de división
        self.divided: bool = False

        # Referencias cardinales a las subinstancias hijas
        self.northwest: Optional[QuadNode] = None
        self.northeast: Optional[QuadNode] = None
        self.southwest: Optional[QuadNode] = None
        self.southeast: Optional[QuadNode] = None

    def subdivide(self) -> None:
        """Divide el nodo en cuatro cuadrantes hijos: NW, NE, SW, SE (Divide)."""
        sub_side = self.boundary.side_length / 2.0
        quarter_side = sub_side / 2.0

        x_mid = self.boundary.x_mid
        y_mid = self.boundary.y_mid
        next_depth = self.depth + 1

        # NW: Cuadrante Noroeste
        boundary_nw = Square(x_mid - quarter_side, y_mid + quarter_side, sub_side)
        self.northwest = QuadNode(boundary_nw, self.capacity, next_depth)

        # NE: Cuadrante Noreste
        boundary_ne = Square(x_mid + quarter_side, y_mid + quarter_side, sub_side)
        self.northeast = QuadNode(boundary_ne, self.capacity, next_depth)

        # SW: Cuadrante Suroeste
        boundary_sw = Square(x_mid - quarter_side, y_mid - quarter_side, sub_side)
        self.southwest = QuadNode(boundary_sw, self.capacity, next_depth)

        # SE: Cuadrante Sureste
        boundary_se = Square(x_mid + quarter_side, y_mid - quarter_side, sub_side)
        self.southeast = QuadNode(boundary_se, self.capacity, next_depth)

        self.divided = True

        # Redistribuir los puntos existentes del nodo actual hacia los subnodos
        existing_points = self.points
        self.points = []
        for pt in existing_points:
            self._insert_into_children(pt)

    def _insert_into_children(self, point: Point) -> bool:
        """Intenta ubicar el punto en uno de los 4 subcuadrantes hijos."""
        for child in (self.northwest, self.northeast, self.southwest, self.southeast):
            if child is not None and child.boundary.contains(point):
                return child.insert(point, max_depth=None)
        return False

    def insert(self, point: Point, max_depth: Optional[int] = None) -> bool:
        """Inserta recursivamente un punto en el árbol (Conquista).

        Args:
            point: Punto geométrico a insertar.
            max_depth: Límite superior de profundidad permitido para prevenir
                sobre-subdivisión en puntos idénticos o hiperagrupados.

        Returns:
            True si el punto fue insertado exitosamente, False si queda fuera.
        """
        # Descartar si el punto no pertenece al bounding box de este nodo
        if not self.boundary.contains(point):
            return False

        # Si no está dividido y tiene capacidad disponible, o alcanzó la profundidad máxima
        reached_depth_limit = max_depth is not None and self.depth >= max_depth
        if (len(self.points) < self.capacity and not self.divided) or reached_depth_limit:
            self.points.append(point)
            return True

        # Si superó la capacidad y aún no está dividido, ejecutar subdivisión
        if not self.divided:
            self.subdivide()

        # Insertar en el subcuadrante correspondiente
        return self._insert_into_children(point)


class Quadtree:
    """Estructura general que administra la raíz y las reglas teóricas del árbol."""

    def __init__(
        self,
        domain_side: float,
        capacity: int = 4,
        min_point_distance: Optional[float] = None,
    ) -> None:
        """Inicializa el Quadtree con el dominio espacial raíz [0, s] x [0, s].

        Args:
            domain_side: Longitud de lado del dominio cuadrado raíz (s = 2^j).
            capacity: Capacidad límite de puntos por nodo antes de subdividir.
            min_point_distance: Distancia mínima entre puntos (c). Si se provee,
                calcula la cota máxima teórica de profundidad: log2(s/c) + 1.5.
        """
        self.side: float = float(domain_side)
        self.capacity: int = int(capacity)
        self.c: Optional[float] = min_point_distance

        # Cuadrado raíz centrado en (side / 2, side / 2) con lado 'side'
        root_boundary = Square(
            x_mid=self.side / 2.0,
            y_mid=self.side / 2.0,
            side_length=self.side,
        )
        self.root: QuadNode = QuadNode(boundary=root_boundary, capacity=self.capacity, depth=0)

        # Cota teórica de profundidad máxima según la formulación del proyecto:
        # Depth_max <= log(s/c) + 3/2
        self.max_theoretical_depth: Optional[int] = self._compute_depth_bound()

    def _compute_depth_bound(self) -> Optional[int]:
        """Calcula la cota teórica superior de profundidad: log2(s/c) + 3/2."""
        if self.c is None or self.c <= 0.0:
            return None
        
        # log2(s / c) + 3/2
        bound_val = math.log2(self.side / self.c) + 1.5
        # Redondear hacia arriba como cota superior entera segura
        return max(1, math.ceil(bound_val))

    def insert(self, point: Point) -> bool:
        """Inserta un punto respetando la cota de profundidad."""
        return self.root.insert(point, max_depth=self.max_theoretical_depth)

    def insert_bulk(self, points: List[Point]) -> int:
        """Inserta una colección de puntos retornando la cantidad de inserciones exitosas."""
        success_count = 0
        for pt in points:
            if self.insert(pt):
                success_count += 1
        return success_count

    def get_max_depth_reached(self, node: Optional[QuadNode] = None) -> int:
        """Recorre el árbol para reportar la profundidad máxima real alcanzada."""
        current = node if node is not None else self.root
        if not current.divided:
            return current.depth

        children = [current.northwest, current.northeast, current.southwest, current.southeast]
        child_depths = [
            self.get_max_depth_reached(child) for child in children if child is not None
        ]
        return max(child_depths) if child_depths else current.depth

ModuleNotFoundError: No module named 'geometry'